In [ ]:
pip install databricks-sdk pyarrow pandas fsspec requests

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()
access_token = os.environ.get("access_token")
host = "dbc-38471c53-2db6.cloud.databricks.com"

TRANSACTION_PATH  = "/Volumes/project_3/datalake/gold_zone/transaction_fact/"
USER_CATALOG_PATH = "/Volumes/project_3/datalake/gold_zone/user_catalog_lookup/"

JOIN_KEY   = "user_id"   # shared column used to join the two tables
OUTPUT_PATH = "./output/gold_zone_combined.parquet"

In [8]:
# ── Helpers: list & read parquet files from a Volume ──────────────────────
import io
import requests
import pyarrow.parquet as pq
import pandas as pd

headers  = {"Authorization": f"Bearer {access_token}"}
base_url = f"https://{host}/api/2.0"

def list_volume_files(path):
    """Return all .parquet file paths inside a Unity Catalog Volume directory."""
    resp = requests.get(
        f"{base_url}/fs/directories{path}",
        headers=headers
    )
    resp.raise_for_status()
    entries = resp.json().get("contents", [])
    return [
        e["path"] for e in entries
        if not e.get("is_directory", False) and e["path"].endswith(".parquet")
    ]

def read_parquet_from_volume(file_path):
    """Stream a single parquet file from a Unity Catalog Volume into a DataFrame."""
    url  = f"https://{host}/api/2.0/fs/files{file_path}"
    resp = requests.get(url, headers=headers, stream=True)
    resp.raise_for_status()
    return pq.read_table(io.BytesIO(resp.content)).to_pandas()

def load_volume(path):
    """Load all parquet files in a Volume path into a single DataFrame."""
    files = list_volume_files(path)
    print(f"  {path}  →  {len(files)} file(s) found")
    if not files:
        return pd.DataFrame()
    return pd.concat(
        [read_parquet_from_volume(f) for f in files],
        ignore_index=True
    )

In [9]:
# ── Load both volumes independently ───────────────────────────────────────
print("Loading volumes...")
df_transactions = load_volume(TRANSACTION_PATH)
df_users        = load_volume(USER_CATALOG_PATH)

print(f"\ntransaction_fact   : {df_transactions.shape}")
print(f"user_catalog_lookup: {df_users.shape}")

print("\ntransaction_fact columns  :", df_transactions.columns.tolist())
print("user_catalog_lookup columns:", df_users.columns.tolist())

Loading volumes...
  /Volumes/project_3/datalake/gold_zone/transaction_fact/  →  12 file(s) found
  /Volumes/project_3/datalake/gold_zone/user_catalog_lookup/  →  2 file(s) found

transaction_fact   : (648609, 12)
user_catalog_lookup: (2000, 9)

transaction_fact columns  : ['transaction_id', 'user_id', 'transaction_type', 'timestamp', 'status', 'payment_method', 'currency', 'subtotal', 'tax', 'total', 'billing_address_id', 'shipping_address_id']
user_catalog_lookup columns: ['user_id', 'email', 'first_name', 'last_name', 'registration_date', 'account_type', 'date_of_birth', 'loyalty_points', 'state']


In [9]:
# ── Join on user_id (left join keeps all transactions) ────────────────────
# Suffix _uc is added to any columns that exist in both tables (except the key)
df_combined = df_transactions.merge(
    df_users,
    on=JOIN_KEY,
    how="left",
    suffixes=("_txn", "_uc")
)

print(f"Combined shape: {df_combined.shape}")
print(f"Unmatched rows (no user_catalog entry): {df_combined[JOIN_KEY].isna().sum()}")
print(df_combined.head(10))

Combined shape: (1297218, 20)
Unmatched rows (no user_catalog entry): 0
                         transaction_id   user_id transaction_type  \
0  e0999e81-8b5e-430d-9770-d154c0ef243d  1c9b9667         purchase   
1  e0999e81-8b5e-430d-9770-d154c0ef243d  1c9b9667         purchase   
2  36f28833-5cb2-4121-979b-425c7762c87f  3ac5bf6d           refund   
3  36f28833-5cb2-4121-979b-425c7762c87f  3ac5bf6d           refund   
4  ed1c62cc-cd4b-410b-874f-11346cc06c1e  87ee137f         purchase   
5  ed1c62cc-cd4b-410b-874f-11346cc06c1e  87ee137f         purchase   
6  a5babd94-b356-4838-be47-391e61112fe6  c6a500ae         purchase   
7  a5babd94-b356-4838-be47-391e61112fe6  c6a500ae         purchase   
8  7e5cddce-a1ff-4571-b769-30c3e0384cce  443e9a69         purchase   
9  7e5cddce-a1ff-4571-b769-30c3e0384cce  443e9a69         purchase   

                     timestamp     status payment_method currency  subtotal  \
0  2026-04-20T18:34:34.021480Z  completed         paypal      USD   2481.85   

In [10]:
# ── Inspect combined schema ────────────────────────────────────────────────
print(df_combined.dtypes)
print(df_combined.shape)

transaction_id                 object
user_id                        object
transaction_type               object
timestamp                      object
status                         object
payment_method                 object
currency                       object
subtotal                      float64
tax                           float64
total                         float64
billing_address_id              int64
shipping_address_id             int64
email                          object
first_name                     object
last_name                      object
registration_date      datetime64[ns]
account_type                   object
date_of_birth          datetime64[ns]
loyalty_points                  int32
state                          object
dtype: object
(1297218, 20)


In [11]:
# ── Save as Parquet for use in other files ─────────────────────────────────
import pyarrow as pa

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

pq.write_table(
    pa.Table.from_pandas(df_combined),
    OUTPUT_PATH,
    compression="snappy"
)

print(f"Saved {len(df_combined):,} rows → {OUTPUT_PATH}")

Saved 1,297,218 rows → ./output/gold_zone_combined.parquet


In [ ]:
# ── How to load this file in any other notebook or script ─────────────────
# import pandas as pd
# df = pd.read_parquet("./output/gold_zone_combined.parquet")
#
# Or with pyarrow:
# import pyarrow.parquet as pq
# df = pq.read_table("./output/gold_zone_combined.parquet").to_pandas()